In [65]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

# Split data into train/val/test (85% / 5% / 10%) just like the baseline paper
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=False 
)

train_data, val_data, test_data = transform(data)

class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Two SAGE layers, each with 64 hidden channels as per Section 4 of the baseline paper
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.normalize(x, p=2, dim=-1) 
        return x

def decode(z, edge_index):
    # Dot product
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    
    z = model(train_data.x.to(device), train_data.edge_index.to(device))
    
    pos_edge_index = train_data.edge_label_index
    neg_edge_index = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge_index.size(1)
    ).to(device)
    
    edge_label_index = torch.cat([pos_edge_index.to(device), neg_edge_index], dim=1)
    edge_label = torch.cat([
        train_data.edge_label.to(device), 
        torch.zeros(neg_edge_index.size(1), device=device)
    ], dim=0)
    
    out = decode(z, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(model, data_split):
    model.eval()
    z = model(data_split.x.to(device), data_split.edge_index.to(device))
    out = decode(z, data_split.edge_label_index.to(device)).sigmoid()
    return roc_auc_score(data_split.edge_label.cpu(), out.cpu()), \
           average_precision_score(data_split.edge_label.cpu(), out.cpu())

# 50 Runs like the baseline paper
num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):
    model = GraphSAGEEncoder(dataset.num_features, 64, 64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(1, 201):
        loss = train(model, optimizer)
        
    test_auc, test_ap = test(model, test_data)
    test_aucs.append(test_auc)
    test_aps.append(test_ap)
    print(f"Run {run:02d} Completed - Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Using device: cuda


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.lo

Run 01 Completed - Test AUC: 0.9153, Test AP: 0.9257
Run 02 Completed - Test AUC: 0.9155, Test AP: 0.9245
Run 03 Completed - Test AUC: 0.9312, Test AP: 0.9335
Run 04 Completed - Test AUC: 0.9192, Test AP: 0.9301
Run 05 Completed - Test AUC: 0.9195, Test AP: 0.9229
Run 06 Completed - Test AUC: 0.9214, Test AP: 0.9273
Run 07 Completed - Test AUC: 0.9219, Test AP: 0.9251
Run 08 Completed - Test AUC: 0.9145, Test AP: 0.9256
Run 09 Completed - Test AUC: 0.9253, Test AP: 0.9299
Run 10 Completed - Test AUC: 0.9256, Test AP: 0.9324
Run 11 Completed - Test AUC: 0.9193, Test AP: 0.9239
Run 12 Completed - Test AUC: 0.9245, Test AP: 0.9257
Run 13 Completed - Test AUC: 0.9222, Test AP: 0.9271
Run 14 Completed - Test AUC: 0.9210, Test AP: 0.9267
Run 15 Completed - Test AUC: 0.9220, Test AP: 0.9293
Run 16 Completed - Test AUC: 0.9210, Test AP: 0.9291
Run 17 Completed - Test AUC: 0.9192, Test AP: 0.9288
Run 18 Completed - Test AUC: 0.9197, Test AP: 0.9258
Run 19 Completed - Test AUC: 0.9160, Test AP: 

In [69]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
dataset = Planetoid(root="data/Planetoid", name="PubMed")

data = dataset[0]
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=False 
)

train_data, val_data, test_data = transform(data)

class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

def decode(z, edge_index):
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    
    z = model(train_data.x.to(device), train_data.edge_index.to(device))
    
    pos_edge_index = train_data.edge_label_index
    neg_edge_index = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge_index.size(1)
    ).to(device)
    
    edge_label_index = torch.cat([pos_edge_index.to(device), neg_edge_index], dim=1)
    edge_label = torch.cat([
        train_data.edge_label.to(device), 
        torch.zeros(neg_edge_index.size(1), device=device)
    ], dim=0)
    
    out = decode(z, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(model, data_split):
    model.eval()
    z = model(data_split.x.to(device), data_split.edge_index.to(device))
    out = decode(z, data_split.edge_label_index.to(device)).sigmoid()
    return roc_auc_score(data_split.edge_label.cpu(), out.cpu()), \
           average_precision_score(data_split.edge_label.cpu(), out.cpu())

num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):
    model = GraphSAGEEncoder(dataset.num_features, 64, 64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(1, 201):
        loss = train(model, optimizer)
        
    test_auc, test_ap = test(model, test_data)
    test_aucs.append(test_auc)
    test_aps.append(test_ap)
    print(f"Run {run:02d} Completed - Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Using device: cuda


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  if osp.exists(f) and torch.lo

Run 01 Completed - Test AUC: 0.9096, Test AP: 0.9175
Run 02 Completed - Test AUC: 0.9036, Test AP: 0.9130
Run 03 Completed - Test AUC: 0.9094, Test AP: 0.9154
Run 04 Completed - Test AUC: 0.9011, Test AP: 0.9111
Run 05 Completed - Test AUC: 0.9112, Test AP: 0.9163
Run 06 Completed - Test AUC: 0.9102, Test AP: 0.9171
Run 07 Completed - Test AUC: 0.9094, Test AP: 0.9135
Run 08 Completed - Test AUC: 0.9057, Test AP: 0.9164
Run 09 Completed - Test AUC: 0.9026, Test AP: 0.9120
Run 10 Completed - Test AUC: 0.9051, Test AP: 0.9160
Run 11 Completed - Test AUC: 0.9055, Test AP: 0.9138
Run 12 Completed - Test AUC: 0.9093, Test AP: 0.9160
Run 13 Completed - Test AUC: 0.9045, Test AP: 0.9111
Run 14 Completed - Test AUC: 0.9070, Test AP: 0.9163
Run 15 Completed - Test AUC: 0.9085, Test AP: 0.9171
Run 16 Completed - Test AUC: 0.9068, Test AP: 0.9158
Run 17 Completed - Test AUC: 0.9076, Test AP: 0.9172
Run 18 Completed - Test AUC: 0.9087, Test AP: 0.9142
Run 19 Completed - Test AUC: 0.9074, Test AP: 

In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
dataset = Planetoid(root="data/Planetoid", name="CiteSeer")

data = dataset[0]
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=False 
)

train_data, val_data, test_data = transform(data)

class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.normalize(x, p=2, dim=-1) 
        return x

def decode(z, edge_index):
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    
    z = model(train_data.x.to(device), train_data.edge_index.to(device))

    pos_edge_index = train_data.edge_label_index
    neg_edge_index = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge_index.size(1)
    ).to(device)
    
    edge_label_index = torch.cat([pos_edge_index.to(device), neg_edge_index], dim=1)
    edge_label = torch.cat([
        train_data.edge_label.to(device), 
        torch.zeros(neg_edge_index.size(1), device=device)
    ], dim=0)
    
    out = decode(z, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(model, data_split):
    model.eval()
    z = model(data_split.x.to(device), data_split.edge_index.to(device))
    out = decode(z, data_split.edge_label_index.to(device)).sigmoid()
    return roc_auc_score(data_split.edge_label.cpu(), out.cpu()), \
           average_precision_score(data_split.edge_label.cpu(), out.cpu())

num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):
    model = GraphSAGEEncoder(dataset.num_features, 64, 64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(1, 201):
        loss = train(model, optimizer)
        
    test_auc, test_ap = test(model, test_data)
    test_aucs.append(test_auc)
    test_aps.append(test_ap)
    print(f"Run {run:02d} Completed - Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")


Using device: cuda


Processing...
Done!
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\io\fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.loa

Run 01 Completed - Test AUC: 0.8908, Test AP: 0.9119
Run 02 Completed - Test AUC: 0.9028, Test AP: 0.9206
Run 03 Completed - Test AUC: 0.8993, Test AP: 0.9166
Run 04 Completed - Test AUC: 0.8961, Test AP: 0.9169
Run 05 Completed - Test AUC: 0.9024, Test AP: 0.9211
Run 06 Completed - Test AUC: 0.9039, Test AP: 0.9211
Run 07 Completed - Test AUC: 0.9016, Test AP: 0.9211
Run 08 Completed - Test AUC: 0.8910, Test AP: 0.9126
Run 09 Completed - Test AUC: 0.9075, Test AP: 0.9229
Run 10 Completed - Test AUC: 0.8941, Test AP: 0.9145
Run 11 Completed - Test AUC: 0.9023, Test AP: 0.9192
Run 12 Completed - Test AUC: 0.9007, Test AP: 0.9217
Run 13 Completed - Test AUC: 0.8987, Test AP: 0.9178
Run 14 Completed - Test AUC: 0.8948, Test AP: 0.9153
Run 15 Completed - Test AUC: 0.8933, Test AP: 0.9137
Run 16 Completed - Test AUC: 0.8919, Test AP: 0.9147
Run 17 Completed - Test AUC: 0.8999, Test AP: 0.9187
Run 18 Completed - Test AUC: 0.9084, Test AP: 0.9212
Run 19 Completed - Test AUC: 0.8887, Test AP: 

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import WikiCS
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

dataset = WikiCS(root="data/WikiCS")
data = dataset[0]

# Wiki-CS contains multiple training/validation masks for node classification.
# For standard Link Prediction, we clear out node classification masks to 
# keep the data object lightweight and avoid indexing bugs during splitting.
del data.train_mask, data.val_mask, data.test_mask

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    add_negative_train_samples=False 
)

train_data, val_data, test_data = transform(data)

class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.normalize(x, p=2, dim=-1) 
        return x

def decode(z, edge_index):
    return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

def train(model, optimizer):
    model.train()
    optimizer.zero_grad()
    
    z = model(train_data.x.to(device), train_data.edge_index.to(device))
    
    pos_edge_index = train_data.edge_label_index
    neg_edge_index = negative_sampling(
        edge_index=train_data.edge_index,
        num_nodes=train_data.num_nodes,
        num_neg_samples=pos_edge_index.size(1)
    ).to(device)
    
    edge_label_index = torch.cat([pos_edge_index.to(device), neg_edge_index], dim=1)
    edge_label = torch.cat([
        train_data.edge_label.to(device), 
        torch.zeros(neg_edge_index.size(1), device=device)
    ], dim=0)
    
    out = decode(z, edge_label_index)
    loss = F.binary_cross_entropy_with_logits(out, edge_label)
    
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(model, data_split):
    model.eval()
    z = model(data_split.x.to(device), data_split.edge_index.to(device))
    out = decode(z, data_split.edge_label_index.to(device)).sigmoid()
    return roc_auc_score(data_split.edge_label.cpu(), out.cpu()), \
           average_precision_score(data_split.edge_label.cpu(), out.cpu())

num_runs = 50
test_aucs = []
test_aps = []

for run in range(1, num_runs + 1):
    model = GraphSAGEEncoder(dataset.num_features, 64, 64).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(1, 201):
        loss = train(model, optimizer)
        
    test_auc, test_ap = test(model, test_data)
    test_aucs.append(test_auc)
    test_aps.append(test_ap)
    print(f"Run {run:02d} Completed - Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")

print(f"\n--- Final Results over {num_runs} runs ---")
print(f"Test AUC: {np.mean(test_aucs):.4f} ± {np.std(test_aucs):.4f}")
print(f"Test AP:  {np.mean(test_aps):.4f} ± {np.std(test_aps):.4f}")

Using device: cuda


C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\datasets\wikics.py:45: UserWarning: The WikiCS dataset now returns an undirected graph by default. Please explicitly specify 'is_undirected=False' to restore the old behavior.
  warnings.warn(
Processing...
Done!
C:\Users\ASUS\AppData\Roaming\Python\Python312\site-packages\torch_geometric\io\fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the use

Run 01 Completed - Test AUC: 0.9454, Test AP: 0.9488
Run 02 Completed - Test AUC: 0.9459, Test AP: 0.9491
Run 03 Completed - Test AUC: 0.9452, Test AP: 0.9482
Run 04 Completed - Test AUC: 0.9459, Test AP: 0.9491
Run 05 Completed - Test AUC: 0.9466, Test AP: 0.9499
Run 06 Completed - Test AUC: 0.9462, Test AP: 0.9494
Run 07 Completed - Test AUC: 0.9462, Test AP: 0.9482
Run 08 Completed - Test AUC: 0.9469, Test AP: 0.9501
Run 09 Completed - Test AUC: 0.9476, Test AP: 0.9506
Run 10 Completed - Test AUC: 0.9430, Test AP: 0.9467
Run 11 Completed - Test AUC: 0.9442, Test AP: 0.9470
Run 12 Completed - Test AUC: 0.9468, Test AP: 0.9495
Run 13 Completed - Test AUC: 0.9482, Test AP: 0.9509
Run 14 Completed - Test AUC: 0.9469, Test AP: 0.9500
Run 15 Completed - Test AUC: 0.9464, Test AP: 0.9493
Run 16 Completed - Test AUC: 0.9463, Test AP: 0.9494
Run 17 Completed - Test AUC: 0.9477, Test AP: 0.9500
Run 18 Completed - Test AUC: 0.9483, Test AP: 0.9517
Run 19 Completed - Test AUC: 0.9478, Test AP: 